In [1]:
import pandas as pd
import numpy as np

In [2]:
goal_scorers = pd.read_csv(r"..\data\goalscorers_processed.csv")
results = pd.read_csv(r"..\data\results_updated.csv")
shootouts = pd.read_csv(r"..\data\shootouts.csv")
former_names = pd.read_csv(r"..\data\former_names.csv")

In [3]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,0


In [4]:
former_names.head()

,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01


In [5]:
results['date'] = pd.to_datetime(results['date'])
former_names['start_date'] = pd.to_datetime(former_names['start_date'])
former_names['end_date'] = pd.to_datetime(former_names['end_date']).fillna(pd.to_datetime('2030-01-01'))

def get_current_name(team_name, match_date):
    # Finding rows where team matches former name and falls within the historical window
    mask = (former_names['former'] == team_name) & \
           (former_names['start_date'] <= match_date) & \
           (former_names['end_date'] >= match_date)
    
    matching_rows = former_names[mask]

    if not matching_rows.empty:
        return matching_rows.iloc[0]['current']
    return team_name

# apply current name to all team names
results['home_team'] = results.apply(lambda row: get_current_name(row['home_team'], row['date']), axis=1)
results['away_team'] = results.apply(lambda row: get_current_name(row['away_team'], row['date']), axis=1)

In [6]:
results['tournament'].unique()

array(['Friendly', 'British Home Championship', 'Évence Coppée Trophy',
       'Muratti Vase', 'Copa Lipton', 'Copa Newton',
       'Copa Premio Honor Argentino', 'Olympic Games',
       'Copa Premio Honor Uruguayo', 'Far Eastern Championship Games',
       'Copa Roca', 'Copa América', 'Inter-Allied Games', 'Peace Cup',
       'Open International Championship', 'Soccer Ashes',
       'Copa Chevallier Boutell', 'Nordic Championship',
       'Central European International Cup', 'Baltic Cup', 'Balkan Cup',
       'Central American and Caribbean Games', 'FIFA World Cup',
       'Copa Rio Branco', 'FIFA World Cup qualification',
       'Bolivarian Games', 'CCCF Championship', 'NAFC Championship',
       'Copa Oswaldo Cruz', 'Asian Games', 'Pan American Championship',
       'Copa del Pacífico', "Copa Bernardo O'Higgins",
       'AFC Asian Cup qualification', 'Atlantic Cup', 'AFC Asian Cup',
       'African Cup of Nations', 'Copa Paz del Chaco',
       'Merdeka Tournament', 'UEFA Euro quali

### Tournament weight system

In [7]:
def assign_tournament_weight(tournament):
    # Standardize string, remove accents, and strip whitespace
    import unicodedata
    t = unicodedata.normalize('NFKD', tournament).encode('ASCII', 'ignore').decode('utf-8').lower().strip()

    # Catch all qualifiers early to prevent leakage into main tournament blocks
    if 'qualification' in t or 'qualifying' in t:
        if 'fifa world cup' in t:
            return 3.0
        if any(x in t for x in ['uefa euro', 'copa america']):
            return 2.0
        if any(x in t for x in ['african cup of nations', 'afc asian cup', 'gold cup', 'concacaf nations league']):
            return 1.5
        return 1.0  # Minor regional qualifiers

    # Tier 1: The Global Elite
    if any(x in t for x in ['fifa world cup', 'confederations cup', 'mundialito']):
        return 4.0
    
    # Tier 2: Top-Tier Continental Elite
    if any(x in t for x in ['uefa euro', 'copa america', 'conmebol-uefa']):
        return 3.0
    
    # Tier 3: Secondary Continental Giants & Olympics
    if any(x in t for x in ['african cup of nations', 'afc asian cup', 'gold cup', 'concacaf championship', 'oceania nations cup', 'olympic games']):
        return 2.5
    
    # Tier 4: Major Continental Leagues & Prestige Regional Cups
    if any(x in t for x in ['uefa nations league', 'concacaf nations league', 'arab cup', 'gulf cup', 'cafa nations cup', 'aff championship', 'asean championship', 'eaff championship', 'dynasty cup']):
        return 2.0
    
    # Tier 5: Minor regional leagues and other tournaments 
    if 'friendly' not in t and not any(x in t for x in ['series', 'challenge cup', 'anniversary']):
        return 1.5
        
    # 7. Tier 6: Friendlies and minor invitational tournament
    return 1.0


In [8]:
results['tournament_weight'] = results['tournament'].apply(assign_tournament_weight)
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight
42314,2019-03-22,Argentina,Venezuela,1.0,3.0,Friendly,Madrid,Spain,1,1.0
45515,2022-09-23,Morocco,Chile,2.0,0.0,Friendly,Barcelona,Spain,1,1.0
12968,1981-09-23,Greece,Sweden,2.0,1.0,Friendly,Salonica,Greece,0,1.0
27861,2004-01-08,Yemen,Saudi Arabia,0.0,2.0,Gulf Cup,Kuwait City,Kuwait,1,2.0
24751,2000-07-16,Zimbabwe,Seychelles,5.0,0.0,African Cup of Nations qualification,Bulawayo,Zimbabwe,0,1.5


### ELO System

In [9]:
results = results.sort_values(by='date').reset_index(drop=True)
results['date'] = pd.to_datetime(results['date'])

# Baseline elo of 1500
elo_ratings = {}
def get_elo(team):
    return elo_ratings.setdefault(team, 1500)

# Elo of teams before the match happens
home_elos = []
away_elos = []

# Initialized beginning year, but will update to row's previous year for elo decay
previous_year = results.loc[0, 'date'].year

# Calculate and update elos for every match
for i, row in results.iterrows():
    home, away = row['home_team'], row['away_team']
    current_year = row['date'].year

    if current_year != previous_year:
        for team in elo_ratings:
            elo_ratings[team] = elo_ratings[team] + 0.1 * (1500 - elo_ratings[team]) # reduction of 10% each year due to outside changes (new season, potential squad and coaching changes)
        
        # updating prevous_year so decay only applies once per year
        previous_year = current_year

    # get elo before match starts
    h_elo = get_elo(home)
    a_elo = get_elo(away)

    home_elos.append(h_elo)
    away_elos.append(a_elo)

    # Expected scores using standard Elo formula
    #       E_A = 1 / (1 + 10^((R_B - R_A) / 400))
    exp_home = 1 / (1 + 10 ** ((a_elo - h_elo) / 400))
    exp_away = 1 - exp_home

    # Outcome weights (Win = 1, Draw = 0.5, Loss = 0)
    if row['home_score'] > row['away_score']:
        actual_home, actual_away = 1.0, 0.0
    elif row['home_score'] < row['away_score']:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # K-Factor to define how volatile rating shifts are, scaled by the tournament weight
    K = 30 * row['tournament_weight']

    # Update ratings
    elo_ratings[home] = h_elo + K * (actual_home - exp_home)
    elo_ratings[away] = a_elo + K * (actual_away - exp_away)

results['home_elo'] = home_elos
results['away_elo'] = away_elos

# ELO-based win probability
results['home_win_prob'] = 1 / (1 + 10 ** ((results['away_elo'] - results['home_elo']) / 400))
results['away_win_prob'] = 1 - results['home_win_prob']

In [10]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,home_elo,away_elo,home_win_prob,away_win_prob
545,1919-03-09,Belgium,France,2.0,2.0,Friendly,Brussels,Belgium,0,1.0,1508.255044,1481.690077,0.538156,0.461844
18539,1992-10-11,Madagascar,Namibia,3.0,0.0,FIFA World Cup qualification,Antananarivo,Madagascar,0,3.0,1553.260410,1459.478987,0.631777,0.368223
17193,1990-03-09,Zambia,Senegal,0.0,0.0,African Cup of Nations,Annaba,Algeria,1,2.5,1578.549756,1641.909021,0.409816,0.590184
39095,2015-09-08,Sweden,Austria,1.0,4.0,UEFA Euro qualification,Solna,Sweden,0,2.0,1611.839396,1705.038353,0.369003,0.630997
36751,2013-03-04,Laos,Sri Lanka,4.0,2.0,AFC Challenge Cup qualification,Vientiane,Laos,0,1.0,1300.905691,1308.355316,0.489281,0.510719


### 5-match history

In [11]:
# result from home team's perspective
results['result'] = np.where(results['home_score'] > results['away_score'], 'W',
                             np.where(results['home_score'] < results['away_score'], 'L', 'D'))

# Rolling win rate
team_matches = pd.concat([
    results[['date', 'home_team', 'result']].rename(columns={'home_team': 'team', 'result': 'res'}),
    results[['date', 'away_team', 'result']].rename(columns={'away_team': 'team', 'result': 'res'})
]).sort_values('date')

# Map results from team's perspective
team_matches['is_win'] = np.where(
    ((team_matches['team'] == results.loc[team_matches.index, 'home_team']) & (team_matches['res'] == 'W')) |
    ((team_matches['team'] == results.loc[team_matches.index, 'away_team']) & (team_matches['res'] == 'L')), 
    1, 0
)

# Calculating rolling 5-match win rate, shifting by 1 to not include current match result
team_matches['rolling_win_rate'] = team_matches.groupby('team')['is_win'].transform(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).fillna(0.33) # filling historical beginnings with even 1/3 split

# Mapping rolling win rates back to main results dataframe
results = results.merge(team_matches[['date', 'team', 'rolling_win_rate']].rename(columns={'rolling_win_rate': 'home_5_game_win_rate'}),
                        left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').drop(columns='team')

results = results.merge(team_matches[['date', 'team', 'rolling_win_rate']].rename(columns={'rolling_win_rate': 'away_5_game_win_rate'}),
                        left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').drop(columns='team')

In [12]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,home_elo,away_elo,home_win_prob,away_win_prob,result,home_5_game_win_rate,away_5_game_win_rate
10700,1976-03-06,Morocco,Nigeria,3.0,1.0,African Cup of Nations,Dire Dawa,Ethiopia,1,2.5,1647.182672,1668.851651,0.468856,0.531144,W,0.6,0.6
4687,1957-09-10,Singapore,Myanmar,2.0,1.0,Friendly,Singapore,Singapore,0,1.0,1489.829487,1474.945131,0.521407,0.478593,W,0.4,0.6
2336,1937-10-13,Czechoslovakia,Latvia,4.0,0.0,Friendly,Prague,Czechoslovakia,0,1.0,1632.356766,1504.819475,0.675718,0.324282,W,0.6,0.4
45202,2022-01-17,Burkina Faso,Ethiopia,1.0,1.0,African Cup of Nations,Bafoussam,Cameroon,1,2.5,1634.188747,1438.540981,0.755144,0.244856,D,0.4,0.2
4173,1954-12-19,India,Myanmar,2.0,1.0,Friendly,Calcutta,India,0,1.0,1566.795567,1523.833056,0.561515,0.438485,W,0.8,0.4


In [13]:
# Difference in elo from home and away teams (positive means home team is favored, negative means away team is favored)
results['home_team_favored'] = results['home_elo'] - results['away_elo']

### 5-match rolling goals scored/conceded

In [14]:
# Add goals to the team_matches
team_goals = pd.concat([
    results[['date', 'home_team', 'home_score', 'away_score']].rename(columns={'home_team': 'team', 'home_score': 'goals_scored', 'away_score': 'goals_conceded'}),
    results[['date', 'away_team', 'away_score', 'home_score']].rename(columns={'away_team': 'team', 'away_score': 'goals_scored', 'home_score': 'goals_conceded'})
]).sort_values('date')

# Calculating 5-match rolling averages (shifted by 1 to avoid current game)
team_goals['rolling_goals_scored'] = team_goals.groupby('team')['goals_scored'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(1.2)
team_goals['rolling_goals_conceded'] = team_goals.groupby('team')['goals_conceded'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()).fillna(1.2)

# Merge back to main results
results = results.merge(team_goals[['date', 'team', 'rolling_goals_scored', 'rolling_goals_conceded']].rename(columns={'rolling_goals_scored': 'home_5_game_goals', 'rolling_goals_conceded': 'home_5_game_conceded'}), left_on=['date', 'home_team'], right_on=['date', 'team'], how='left').drop(columns='team')
results = results.merge(team_goals[['date', 'team', 'rolling_goals_scored', 'rolling_goals_conceded']].rename(columns={'rolling_goals_scored': 'away_5_game_goals', 'rolling_goals_conceded': 'away_5_game_conceded'}), left_on=['date', 'away_team'], right_on=['date', 'team'], how='left').drop(columns='team')

In [15]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_win_prob,away_win_prob,result,home_5_game_win_rate,away_5_game_win_rate,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded
24468,1980-01-19,Congo,Ivory Coast,2.0,0.0,Friendly,Brazzaville,Congo,0,1.0,...,0.452342,0.547658,W,0.6,0.6,-33.216718,1.8,2.0,2.0,0.8
21517,1975-05-17,Germany,Netherlands,1.0,1.0,Friendly,Frankfurt,Germany,0,1.0,...,0.555129,0.444871,D,0.4,0.8,38.464193,1.2,1.2,2.4,0.8
35115,1997-03-12,Cyprus,Greece,0.0,4.0,Friendly,Paralímni,Cyprus,0,1.0,...,0.364619,0.635381,L,0.6,0.2,-96.477955,1.6,1.2,1.2,0.8
26842,1983-09-14,New Caledonia,Wallis Islands and Futuna,4.0,0.0,South Pacific Games,Apia,Western Samoa,1,1.5,...,0.579040,0.420960,W,0.6,0.4,55.387360,2.6,2.2,1.0,2.0
29621,1988-10-19,Italy,Norway,2.0,1.0,Friendly,Pescara,Italy,0,1.0,...,0.825715,0.174285,W,0.4,0.0,270.227510,0.8,0.8,0.6,1.0


### Historical Head-to-Head Advantage or Disadvantage

In [16]:
def calculate_h2h(df):
    # Sort names by home_team, away_team to compared historical matchups against the same teams
    teams = np.sort(df[['home_team', 'away_team']].values, axis=1)
    df['matchup_key'] = teams[:, 0] + "_" + teams[:, 1]

    # Tracking wins for team that appears first in the key
    df['team_a_won'] = np.where(
        ((df['home_team'] == df['matchup_key'].str.split('_').str[0]) & (df['result'] == 'W')) |
        ((df['away_team'] == df['matchup_key'].str.split('_').str[0]) & (df['result'] == 'L')), 
        1, 0
    )

    # Calculating win rate of Team A over Team B
    df['h2h_team_a_win_rate'] = df.groupby('matchup_key')['team_a_won'].transform(lambda x: x.shift(1).expanding().mean().fillna(0.5))

    # Translate this back to the perspective of the current home_team
    df['home_historical_advantage'] = np.where(df['home_team'] == df['matchup_key'].str.split('_').str[0], 
                                        df['h2h_team_a_win_rate'], 
                                        1 - df['h2h_team_a_win_rate'])
    
    return df.drop(columns=['matchup_key', 'team_a_won', 'h2h_team_a_win_rate'])

results = calculate_h2h(results)

In [17]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,away_win_prob,result,home_5_game_win_rate,away_5_game_win_rate,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded,home_historical_advantage
37468,1999-02-28,British Virgin Islands,Antigua and Barbuda,0.0,2.0,Friendly,Road Town,British Virgin Islands,0,1.0,...,0.557632,L,0.4,0.2,-40.225541,1.4,2.6,0.6,1.0,0.000000
25394,1981-09-16,South Korea,Thailand,1.0,1.0,Merdeka Tournament,Kuala Lumpur,Malaysia,1,1.5,...,0.212951,D,0.8,0.6,227.089350,1.6,0.2,1.8,2.2,0.750000
16716,1973-04-08,Tanzania,Ethiopia,3.0,0.0,African Cup of Nations qualification,Dar es Salaam,Tanzania,0,1.5,...,0.617534,W,0.0,0.2,-83.227418,1.8,1.2,1.2,0.8,0.938462
32955,1994-04-03,Zambia,Senegal,1.0,0.0,African Cup of Nations,Sousse,Tunisia,1,2.5,...,0.230492,W,0.6,0.2,209.422993,1.4,0.6,0.4,0.4,1.000000
25956,1983-01-17,Benin,Mauritania,1.0,1.0,Friendly,Porto-Novo,Benin,0,1.0,...,0.438276,D,0.2,0.2,43.110409,1.4,2.0,1.4,2.8,1.000000


### 5-game Goal Differential

In [18]:
results['home_5_game_goal_differential'] = results['home_5_game_goals'] - results['home_5_game_conceded']
results['away_5_game_goal_differential'] = results['away_5_game_goals'] - results['away_5_game_conceded']

In [19]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_5_game_win_rate,away_5_game_win_rate,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded,home_historical_advantage,home_5_game_goal_differential,away_5_game_goal_differential
18630,1973-08-03,Malaysia,Cambodia,1.0,0.0,Merdeka Tournament,Kuala Lumpur,Malaysia,0,1.5,...,0.2,0.4,8.054278,1.0,1.2,0.8,0.8,0.988235,-0.2,0.0
16726,1973-04-08,Tanzania,Ethiopia,3.0,0.0,African Cup of Nations qualification,Dar es Salaam,Tanzania,0,1.5,...,0.0,0.4,-83.227418,0.6,2.0,1.6,1.4,0.946667,-1.4,0.2
12029,1966-08-20,Singapore,Taiwan,4.0,1.0,Merdeka Tournament,Ipoh,Malaysia,1,1.5,...,0.2,0.2,-75.697560,1.0,1.8,1.0,2.4,0.973684,-0.8,-1.4
4032,1927-12-10,Chile,Uruguay,2.0,3.0,Friendly,Viña del Mar,Chile,0,1.0,...,0.4,0.6,-220.854074,3.0,1.8,3.2,0.8,0.000000,1.2,2.4
16348,1972-07-02,DR Congo,Guinea,6.0,0.0,Friendly,Kinshasa,Zaïre,0,1.0,...,0.2,0.4,9.470560,2.0,2.0,2.0,1.0,0.000000,0.0,1.0


In [20]:
results.to_csv(r"..\data\results_engineered.csv", index=False)

# Shootouts

In [21]:
shootouts.head()

,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


In [22]:
shootouts['date'] = pd.to_datetime(shootouts['date'])
shootouts['home_team'] = shootouts.apply(lambda row: get_current_name(row['home_team'], row['date']), axis=1)
shootouts['away_team'] = shootouts.apply(lambda row: get_current_name(row['away_team'], row['date']), axis=1)
shootouts['winner'] = shootouts.apply(lambda row: get_current_name(row['winner'], row['date']), axis=1)

# shootouts by date and matchup
team_shootouts = pd.concat([
    shootouts[['date', 'home_team', 'winner']].rename(columns={'home_team': 'team'}),
    shootouts[['date', 'away_team', 'winner']].rename(columns={'away_team': 'team'})
]).sort_values('date')

# check if team won the shootout
team_shootouts['is_shootout_win'] = (team_shootouts['team'] == team_shootouts['winner']).astype(int)

# historical matchup penalty win rate
team_shootouts['historical_matchup_penalty_win_rate'] = (
    team_shootouts.groupby('team')['is_shootout_win']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.50) # If a team has never been in a shootout, default to a 50/50 baseline
)

team_shootouts = team_shootouts.drop_duplicates(subset=['date', 'team'], keep='last')

# mapping (date, team) -> win rate
shootout_map = dict(zip(zip(team_shootouts['date'], team_shootouts['team']), team_shootouts['historical_matchup_penalty_win_rate']))

# Use a zip layout to map the keys directly into new columns
results['home_historical_matchup_pen_win_rate'] = pd.Series(zip(results['date'], results['home_team'])).map(shootout_map)
results['away_historical_matchup_pen_win_rate'] = pd.Series(zip(results['date'], results['away_team'])).map(shootout_map)

# fill teams that have zero shootout history with the 50/50 baseline
results['home_historical_matchup_pen_win_rate'] = results['home_historical_matchup_pen_win_rate'].fillna(0.50)
results['away_historical_matchup_pen_win_rate'] = results['away_historical_matchup_pen_win_rate'].fillna(0.50)

In [23]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_team_favored,home_5_game_goals,home_5_game_conceded,away_5_game_goals,away_5_game_conceded,home_historical_advantage,home_5_game_goal_differential,away_5_game_goal_differential,home_historical_matchup_pen_win_rate,away_historical_matchup_pen_win_rate
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,0,1.0,...,0.000000,1.200000,1.200000,1.200000,1.200000,0.500000,0.000000,0.000000,0.5,0.5
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,0,1.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5,0.5
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,0,1.0,...,-27.000000,1.000000,2.000000,2.000000,1.000000,0.500000,-1.000000,1.000000,0.5,0.5
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,0,1.0,...,-4.794017,1.666667,1.333333,1.333333,1.666667,0.333333,0.333333,-0.333333,0.5,0.5
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,0,1.0,...,3.942085,1.500000,1.750000,1.750000,1.500000,0.750000,-0.250000,0.250000,0.5,0.5


### Psychological impact on performance based on order of penalties taken

In [24]:
shootouts['date'] = pd.to_datetime(shootouts['date'])

# Only apply name mapping where first_shooter is NOT null
not_null_mask = shootouts['first_shooter'].notna()
shootouts.loc[not_null_mask, 'first_shooter'] = shootouts[not_null_mask].apply(
    lambda row: get_current_name(row['first_shooter'], row['date']), axis=1
)

# If first_shooter is NaN, second_shooter becomes NaN
shootouts['second_shooter'] = np.where(
    shootouts['first_shooter'].isna(), 
    np.nan, 
    np.where(shootouts['first_shooter'] == shootouts['home_team'], shootouts['away_team'], shootouts['home_team'])
)

# Drop rows where no shootout happened before stacking historical records
valid_shootouts = shootouts.dropna(subset=['first_shooter', 'second_shooter'])

# Tracking performance when shooting first
first_shooters = valid_shootouts[['date', 'first_shooter', 'winner']].rename(columns={'first_shooter': 'team'})
first_shooters['won_as_first'] = (first_shooters['team'] == first_shooters['winner']).astype(int)
first_shooters['historical_first_kick_clutch'] = (
    first_shooters.groupby('team')['won_as_first']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.60)
)

# Track performance when shooting second
second_shooters = valid_shootouts[['date', 'second_shooter', 'winner']].rename(columns={'second_shooter': 'team'})
second_shooters['won_as_second'] = (second_shooters['team'] == second_shooters['winner']).astype(int)
second_shooters['historical_second_kick_resilience'] = (
    second_shooters.groupby('team')['won_as_second']
    .transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.40)
)

# Drop duplicates to create clean daily lookups
first_lookup = first_shooters.drop_duplicates(subset=['date', 'team'], keep='last')
second_lookup = second_shooters.drop_duplicates(subset=['date', 'team'], keep='last')

# Creating maps for first and second penalty shooters
first_map = dict(zip(zip(first_lookup['date'], first_lookup['team']), first_lookup['historical_first_kick_clutch']))
second_map = dict(zip(zip(second_lookup['date'], second_lookup['team']), second_lookup['historical_second_kick_resilience']))

# Map to main results dataset
results['home_pen_win_rate_shot_first'] = pd.Series(zip(results['date'], results['home_team'])).map(first_map).fillna(0.60)
results['away_pen_win_rate_shot_first'] = pd.Series(zip(results['date'], results['away_team'])).map(first_map).fillna(0.60)

results['home_pen_win_rate_shot_second'] = pd.Series(zip(results['date'], results['home_team'])).map(second_map).fillna(0.40)
results['away_pen_win_rate_shot_second'] = pd.Series(zip(results['date'], results['away_team'])).map(second_map).fillna(0.40)

In [25]:
shootouts.head()

,date,home_team,away_team,winner,first_shooter,second_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN,NaN


In [26]:
shootouts.sample(5)

,date,home_team,away_team,winner,first_shooter,second_shooter
478,2016-05-27,Corsica,Basque Country,Basque Country,NaN,NaN
298,2002-01-27,Mexico,South Korea,South Korea,South Korea,Mexico
561,2020-10-08,Scotland,Israel,Scotland,Scotland,Israel
144,1990-08-03,China,South Korea,South Korea,NaN,NaN
272,2000-05-21,Barbados,Cuba,Barbados,NaN,NaN


In [27]:
results.sample(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,away_5_game_conceded,home_historical_advantage,home_5_game_goal_differential,away_5_game_goal_differential,home_historical_matchup_pen_win_rate,away_historical_matchup_pen_win_rate,home_pen_win_rate_shot_first,away_pen_win_rate_shot_first,home_pen_win_rate_shot_second,away_pen_win_rate_shot_second
38500,2000-02-20,Costa Rica,Trinidad and Tobago,1.0,2.0,Gold Cup,San Diego,United States,1,2.5,...,1.8,0.714286,0.4,-0.8,0.5,0.5,0.6,0.6,0.4,0.4
47961,2010-01-11,Ivory Coast,Burkina Faso,0.0,0.0,African Cup of Nations,Cabinda,Angola,1,2.5,...,2.4,0.894737,1.2,-1.4,0.5,0.5,0.6,0.6,0.4,0.4
27852,1985-02-27,El Salvador,Suriname,3.0,0.0,CONCACAF Championship,San Salvador,El Salvador,0,2.5,...,1.2,0.750000,0.2,-0.4,0.5,0.5,0.6,0.6,0.4,0.4
33343,1994-11-12,United Arab Emirates,Bahrain,0.0,0.0,Gulf Cup,Abu Dhabi,United Arab Emirates,0,2.0,...,2.0,0.454545,0.8,-0.4,0.5,0.5,0.6,0.6,0.4,0.4
13396,1969-02-02,Uganda,Cameroon,2.0,1.0,African Cup of Nations qualification,Kampala,Uganda,0,1.5,...,2.2,1.000000,-0.8,-0.8,0.5,0.5,0.6,0.6,0.4,0.4
58779,2021-09-02,South Korea,Iraq,0.0,0.0,FIFA World Cup qualification,Seoul,South Korea,0,3.0,...,0.8,0.947368,1.8,1.4,0.5,0.5,0.6,0.6,0.4,0.4
16412,1972-08-29,United States,Canada,2.0,2.0,CONCACAF Championship qualification,Baltimore,United States,0,1.0,...,1.2,0.454545,0.4,0.2,0.5,0.5,0.6,0.6,0.4,0.4
6864,1939-06-04,Yugoslavia,Italy,1.0,2.0,Friendly,Belgrade,Yugoslavia,0,1.0,...,1.2,0.000000,-0.2,1.0,0.5,0.5,0.6,0.6,0.4,0.4
57595,2019-10-15,Mauritania,Libya,0.0,0.0,Friendly,Nouakchott,Mauritania,0,1.0,...,1.0,0.500000,-1.2,1.4,0.5,0.5,0.6,0.6,0.4,0.4
33149,1994-07-09,Netherlands,Brazil,2.0,3.0,FIFA World Cup,Dallas,United States,1,4.0,...,0.2,0.666667,1.2,2.0,0.5,0.5,0.6,0.6,0.4,0.4


In [28]:
results['home_pen_win_rate_shot_first'].unique()

array([0.6       , 1.        , 0.        , 0.5       , 0.33333333,
       0.66666667, 0.25      , 0.75      , 0.8       , 0.57142857,
       0.625     , 0.71428571, 0.375     , 0.4       ])

In [29]:
results['away_pen_win_rate_shot_first'].unique()

array([0.6       , 1.        , 0.        , 0.75      , 0.33333333,
       0.5       , 0.66666667, 0.4       , 0.8       , 0.42857143,
       0.25      , 0.55555556, 0.44444444])

In [30]:
results['home_pen_win_rate_shot_second'].unique()

array([0.4       , 0.        , 0.5       , 0.33333333, 1.        ,
       0.25      , 0.2       , 0.71428571, 0.66666667, 0.6       ,
       0.75      , 0.42857143])

In [31]:
results['away_pen_win_rate_shot_second'].unique()

array([0.4       , 1.        , 0.        , 0.33333333, 0.5       ,
       0.25      , 0.66666667, 0.75      , 0.8       , 0.6       ,
       0.63636364, 0.42857143, 0.2       , 0.375     ])

In [32]:
results.to_csv(r"..\data\results_engineered.csv", index=False)

In [33]:
results = pd.read_csv(r"..\data\results_engineered.csv")

In [34]:
goal_scorers.head()

,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,0,0
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,0,0
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,0,0
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,0,0
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,0,0


In [35]:
goal_scorers.sample(5)

,date,home_team,away_team,team,scorer,minute,own_goal,penalty
23675,2004-06-02,Brazil,Argentina,Brazil,Ronaldo,16.0,0,1
35262,2015-11-12,Afghanistan,Cambodia,Afghanistan,Mustafa Zazai,42.0,0,0
36408,2016-11-11,Czech Republic,Norway,Norway,Joshua King,87.0,0,0
23050,2003-10-11,Cyprus,Slovenia,Cyprus,Stavros Georgiou,71.0,0,0
20428,2001-02-19,Bangladesh,Mongolia,Mongolia,Boldyn Buman-Uchral,90.0,0,0


In [36]:
# percentage of goals in clutch time 
len(goal_scorers[goal_scorers['minute'] >= 75]) / len(goal_scorers)

0.23267564463931517

In [37]:
# percentage of own goals
len(goal_scorers[goal_scorers['own_goal'] == 1]) / len(goal_scorers)

0.019605386783589102

In [38]:
results['date'] = pd.to_datetime(results['date'])
results = results.sort_values('date').reset_index(drop=True)

goal_scorers['date'] = pd.to_datetime(goal_scorers['date'])
goal_scorers['team'] = goal_scorers.apply(lambda row: get_current_name(row['team'], row['date']), axis=1)
goal_scorers = goal_scorers.sort_values('date').reset_index(drop=True)

# tracking goals in the clutch time and if a goal scored was an own goal
goal_scorers['is_late_goal'] = (goal_scorers['minute'] >= 75).astype(int)
goal_scorers['is_own_goal'] = (goal_scorers['own_goal'] == True).astype(int)

# historical averages for clutch goals
goal_scorers['hist_clutch_scoring_rate'] = (
    goal_scorers.groupby('team')['is_late_goal'].transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.23) # Global baseline: roughly 23% of goals naturally occur late game
)

goal_scorers['hist_opponent_own_goal_rate'] = (
    goal_scorers.groupby('team')['is_own_goal'].transform(lambda x: x.shift(1).expanding().mean())
    .fillna(0.02) # Global baseline: roughly 2% of goals are opponent own goals
)

# Drop duplicate rows for the same date and team, keeping the latest pre-match baseline
lookup_df = goal_scorers[['date', 'team', 'hist_clutch_scoring_rate', 'hist_opponent_own_goal_rate']].drop_duplicates(subset=['date', 'team'], keep='last')
lookup_df = lookup_df.sort_values('date').reset_index(drop=True)

# merge data for home and away teams
results = pd.merge_asof(
    results, 
    lookup_df.rename(columns={'team': 'home_team', 'hist_clutch_scoring_rate': 'home_clutch_scoring_rate', 'hist_opponent_own_goal_rate': 'home_og_inducer_rate'}),
    on='date', 
    by='home_team', 
    direction='backward'
)

# Merge for Away Team
results = pd.merge_asof(
    results, 
    lookup_df.rename(columns={'team': 'away_team', 'hist_clutch_scoring_rate': 'away_clutch_scoring_rate', 'hist_opponent_own_goal_rate': 'away_og_inducer_rate'}),
    on='date', 
    by='away_team', 
    direction='backward'
)

results[['home_clutch_scoring_rate', 'away_clutch_scoring_rate']] = results[['home_clutch_scoring_rate', 'away_clutch_scoring_rate']].fillna(0.23)
results[['home_og_inducer_rate', 'away_og_inducer_rate']] = results[['home_og_inducer_rate', 'away_og_inducer_rate']].fillna(0.02)

In [39]:
results.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,tournament_weight,...,home_historical_matchup_pen_win_rate,away_historical_matchup_pen_win_rate,home_pen_win_rate_shot_first,away_pen_win_rate_shot_first,home_pen_win_rate_shot_second,away_pen_win_rate_shot_second,home_clutch_scoring_rate,home_og_inducer_rate,away_clutch_scoring_rate,away_og_inducer_rate
28191,1985-09-01,Costa Rica,Canada,0.0,0.0,CONCACAF Championship,San José,Costa Rica,0,2.5,...,0.5,0.5,0.6,0.6,0.4,0.4,0.175439,0.008772,0.216216,0.000000
39230,2000-10-07,Andorra,Estonia,1.0,2.0,FIFA World Cup qualification,Sant Julià de Lòria,Andorra,0,3.0,...,0.5,0.5,0.6,0.6,0.4,0.4,0.200000,0.000000,0.179487,0.012821
47981,2010-01-20,Chile,Panama,2.0,1.0,Friendly,Coquimbo,Chile,0,1.0,...,0.5,0.5,0.6,0.6,0.4,0.4,0.200445,0.011136,0.223684,0.013158
48250,2010-06-14,Italy,Paraguay,1.0,1.0,FIFA World Cup,Cape Town,South Africa,1,4.0,...,0.5,0.5,0.6,0.6,0.4,0.4,0.233918,0.025341,0.230594,0.018265
44874,2006-11-27,Somalia,Rwanda,0.0,3.0,CECAFA Cup,Addis Ababa,Ethiopia,1,1.5,...,0.5,0.5,0.6,0.6,0.4,0.4,0.230000,0.020000,0.133333,0.000000


In [40]:
results.to_csv(r"..\data\results_engineered.csv", index=False)